# Detección de Lavado de Dinero  
## Limpieza adicional, exclusión de artefacto y preparación PyTorch

1. Documentamos y **excluimos un artefacto del simulador PaySim** detectado al correr la Parte 1 contra el dataset completo.
2. Hacemos verificaciones de limpieza adicionales (duplicados, montos inválidos, la columna `isFlaggedFraud`, outliers).
3. Reconstruimos las secuencias con el set de features corregido.
4. Preparamos el `Dataset`/`DataLoader` de PyTorch, listos para el Componente 3 (modelo).


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DATA_PATH = "PS_20174392719_1491204439457_log.csv"  

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

Shape: (6362620, 11)


## 1. Artefacto de balance de destino: por qué lo excluimos

Al correr la Parte 1 contra el dataset completo (6.36M filas), `errorBalanceDest` mostró una separación sospechosamente perfecta entre fraude y no-fraude. La causa: PaySim no actualiza consistentemente `oldbalanceDest`/`newbalanceDest` para transacciones legítimas de ciertos tipos, dejándolas en 0 aunque sí hubo movimiento de dinero — es un defecto de cómo se generó el dataset sintético, no un patrón de comportamiento real de lavado.



In [2]:
zero_dest_rate_by_fraud = df[df['nameDest'].str.startswith('C')].groupby('isFraud').apply(
    lambda g: ((g['oldbalanceDest'] == 0) & (g['newbalanceDest'] == 0)).mean()
)
print("Tasa de 'balance destino en cero' por clase:")
print(zero_dest_rate_by_fraud)
print(f"\nRatio fraude/no-fraude: {zero_dest_rate_by_fraud[1] / zero_dest_rate_by_fraud[0]:.1f}x")

Tasa de 'balance destino en cero' por clase:
isFraud
0    0.038476
1    0.496286
dtype: float64

Ratio fraude/no-fraude: 12.9x


C:\Users\olivi\AppData\Local\Temp\ipykernel_2664\2631853259.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  zero_dest_rate_by_fraud = df[df['nameDest'].str.startswith('C')].groupby('isFraud').apply(


**Decisión: excluimos `oldbalanceDest`, `newbalanceDest` y `errorBalanceDest`** (tanto en su forma cruda como transformada) del set de features. Las dejamos documentadas aquí y en el dataset crudo por transparencia, pero no entran al modelo. La razón es explícita: si el modelo aprende a explotar `oldbalanceDest == newbalanceDest == 0` en vez de patrones de comportamiento (montos, frecuencia, tipo, drenado de balance de origen), va a lucir excelente en este dataset sintético y no va a generalizar a datos reales de un banco, donde ese defecto de tracking de balances no existe. Buscamos que el modelo aprenda señales más realistas y transferibles, aunque eso signifique sacrificar algo de desempeño aparente en PaySim.

Las features de balance del **origen** (`oldbalanceOrg`, `newbalanceOrig`, `errorBalanceOrig`) sí se mantienen: reflejan el patrón de "drenado completo de cuenta" (`amount ≈ oldbalanceOrg`), que sí es un comportamiento genuino simulado (y consistente con patrones reales de lavado: mover el saldo completo de una cuenta).

## 2. Verificaciones de limpieza adicionales

In [3]:
print("Montos negativos:", (df['amount'] < 0).sum())
print("Montos en cero:", (df['amount'] == 0).sum())
print("Filas duplicadas:", df.duplicated().sum())
print("\nDistribución de isFlaggedFraud:")
print(df['isFlaggedFraud'].value_counts())
print("\nCruce isFraud x isFlaggedFraud:")
print(pd.crosstab(df['isFraud'], df['isFlaggedFraud']))
print("\nPercentiles de amount:", df['amount'].quantile([0.5, 0.9, 0.99, 0.999, 1.0]).to_dict())

Montos negativos: 0
Montos en cero: 16
Filas duplicadas: 0

Distribución de isFlaggedFraud:
isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

Cruce isFraud x isFlaggedFraud:
isFlaggedFraud        0   1
isFraud                    
0               6354407   0
1                  8197  16

Percentiles de amount: {0.5: 74871.94, 0.9: 365423.30900000007, 0.99: 1615979.4715999917, 0.999: 8956797.676964078, 1.0: 92445516.64}


**Montos:** no hay negativos ni ceros — no se requiere limpieza de valores inválidos.

**Duplicados:** ninguno en el sample — se deja el check en el notebook para correr también contra el dataset completo.

**`isFlaggedFraud`:** es la regla de negocio simple que trae el propio simulador (marca transferencias que superan un umbral fijo). Es de público conocimiento en la documentación de PaySim que esta bandera casi nunca se activa y tiene recall pésimo sobre el fraude real — en este sample ni siquiera se activó ninguna vez. **La excluimos como feature**: no aporta señal aprendible (varianza ~0) y, de tener algún valor, sería el de una regla ya conocida y trivial de aplicar por separado, no algo que un modelo de secuencia deba redescubrir. *(No pudimos confirmar la tasa exacta de activación en el dataset completo desde este sample — si te interesa, `df['isFlaggedFraud'].sum()` sobre el CSV completo lo confirma.)*

**Outliers de `amount`:** la cola es pesada (P99.9 ya en millones) pero son montos legítimos dentro del rango de lo que un simulador financiero genera, y los montos grandes son precisamente relevantes para lavado — no truncamos ni hacemos winsorizing. El `log1p` ya aplicado en la Parte 1 es suficiente para que la escala no domine el entrenamiento.

## 3. Reconstrucción de secuencias con el set de features corregido

Mismo pipeline vectorizado de la Parte 1 (filtrado de merchants, log-transform, `type_idx`, `delta_t`, split por grupo 70/15/15, normalización ajustada solo con train), pero **sin** las columnas de balance destino.

**Corrección de un bug detectado al correr contra el dataset completo:** `type_idx` no debe pasar por el `StandardScaler` — es una categoría (0-3), no una magnitud continua. Escalarla la convierte en un z-score que, al castearse de vuelta a entero para el `nn.Embedding` del modelo, corrompe la categoría (y puede caer fuera de rango, rompiendo el embedding). En el sample no producía error porque los valores truncados por casualidad quedaban dentro de rango, pero sí corrompía la señal de tipo silenciosamente. Ya corregido: `SCALE_COLS` excluye `type_idx` explícitamente.

In [4]:
FEATURE_COLS = ['log_amount', 'log_oldbalanceOrg', 'log_newbalanceOrig',
                'errorBalanceOrig', 'delta_t', 'type_idx']  # <- sin balance destino / errorBalanceDest

df_c = df[df['nameDest'].str.startswith('C')].copy()

for col in ['amount', 'oldbalanceOrg', 'newbalanceOrig']:
    df_c[f'log_{col}'] = np.log1p(df_c[col])
df_c['errorBalanceOrig'] = df_c['newbalanceOrig'] + df_c['amount'] - df_c['oldbalanceOrg']
type_map = {t: i for i, t in enumerate(sorted(df_c['type'].unique()))}
df_c['type_idx'] = df_c['type'].map(type_map)
df_c = df_c.sort_values(['nameDest', 'step']).reset_index(drop=True)
df_c['delta_t'] = df_c.groupby('nameDest')['step'].diff().fillna(0)

seq_lengths = df_c.groupby('nameDest').size()
seq_labels = df_c.groupby('nameDest')['isFraud'].max()
MAX_LEN = int(seq_lengths.quantile(0.95))
print("MAX_LEN (P95):", MAX_LEN, "| cuentas:", len(seq_labels), "| positivos:", f"{seq_labels.mean():.2%}")

dests = seq_labels.index.values
labels = seq_labels.values
TEST_FRAC, VAL_FRAC = 0.15, 0.15
gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=RANDOM_SEED)
trainval_idx, test_idx = next(gss1.split(dests, labels, groups=dests))
trainval_dests, test_dests = dests[trainval_idx], dests[test_idx]
val_frac_within_trainval = VAL_FRAC / (1 - TEST_FRAC)
gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac_within_trainval, random_state=RANDOM_SEED)
tv_labels = seq_labels.loc[trainval_dests].values
train_idx, val_idx = next(gss2.split(trainval_dests, tv_labels, groups=trainval_dests))
train_dests, val_dests = trainval_dests[train_idx], trainval_dests[val_idx]
print(f"Train: {len(train_dests)} | Val: {len(val_dests)} | Test: {len(test_dests)}")

SCALE_COLS = [c for c in FEATURE_COLS if c != 'type_idx']  # type_idx es categorico: nunca se escala (ver nota mas abajo)
scaler = StandardScaler().fit(df_c.loc[df_c['nameDest'].isin(train_dests), SCALE_COLS])
df_c[SCALE_COLS] = scaler.transform(df_c[SCALE_COLS])

def build_sequence_tensors_vectorized(dest_ids_subset, df_full, feature_cols, max_len):
    sub = df_full[df_full['nameDest'].isin(dest_ids_subset)].copy()
    sub['acc_idx'], uniques = pd.factorize(sub['nameDest'], sort=False)
    n_accounts = len(uniques)
    labels_by_acc = sub.groupby('acc_idx')['isFraud'].max().reindex(range(n_accounts)).values
    rank_asc = sub.groupby('acc_idx').cumcount()
    group_size = sub.groupby('acc_idx')['acc_idx'].transform('size')
    rank_from_end = (group_size - 1 - rank_asc).values
    keep = rank_from_end < max_len
    col_pos = max_len - 1 - rank_from_end[keep]
    row_acc = sub['acc_idx'].values[keep]
    X = np.zeros((n_accounts, max_len, len(feature_cols)), dtype=np.float32)
    mask = np.zeros((n_accounts, max_len), dtype=np.float32)
    X[row_acc, col_pos, :] = sub.loc[keep, feature_cols].values
    mask[row_acc, col_pos] = 1.0
    y = labels_by_acc.astype(np.float32)
    return X, mask, y

X_train, mask_train, y_train = build_sequence_tensors_vectorized(train_dests, df_c, FEATURE_COLS, MAX_LEN)
X_val, mask_val, y_val = build_sequence_tensors_vectorized(val_dests, df_c, FEATURE_COLS, MAX_LEN)
X_test, mask_test, y_test = build_sequence_tensors_vectorized(test_dests, df_c, FEATURE_COLS, MAX_LEN)
print("shapes:", X_train.shape, X_val.shape, X_test.shape, "| features:", FEATURE_COLS)

MAX_LEN (P95): 24 | cuentas: 571961 | positivos: 1.43%
Train: 400372 | Val: 85794 | Test: 85795
shapes: (400372, 24, 6) (85794, 24, 6) (85795, 24, 6) | features: ['log_amount', 'log_oldbalanceOrg', 'log_newbalanceOrig', 'errorBalanceOrig', 'delta_t', 'type_idx']


## 4. `Dataset` y `DataLoader` de PyTorch

- `type_idx` se separa del resto de features numéricas porque en el modelo (Componente 3) va a pasar por una capa `nn.Embedding`, no por la misma normalización que las demás.
- `WeightedRandomSampler` **solo en train**, con pesos inversamente proporcionales a la frecuencia de clase — val/test mantienen la distribución real.

In [5]:
class AMLSequenceDataset(Dataset):
    def __init__(self, X, mask, y, type_col_idx):
        # separamos la columna categórica (type_idx) del resto de features numéricas
        num_cols = [i for i in range(X.shape[-1]) if i != type_col_idx]
        self.X_num = torch.tensor(X[:, :, num_cols], dtype=torch.float32)
        self.X_type = torch.tensor(X[:, :, type_col_idx], dtype=torch.long)
        self.mask = torch.tensor(mask, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "x_num": self.X_num[idx],
            "x_type": self.X_type[idx],
            "mask": self.mask[idx],
            "y": self.y[idx],
        }

TYPE_COL_IDX = FEATURE_COLS.index('type_idx')

train_ds = AMLSequenceDataset(X_train, mask_train, y_train, TYPE_COL_IDX)
val_ds = AMLSequenceDataset(X_val, mask_val, y_val, TYPE_COL_IDX)
test_ds = AMLSequenceDataset(X_test, mask_test, y_test, TYPE_COL_IDX)

pos = y_train.sum(); neg = len(y_train) - pos
sample_weights = np.where(y_train == 1, 1.0 / max(pos, 1), 1.0 / max(neg, 1))
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

pos_weight = torch.tensor(neg / max(pos, 1), dtype=torch.float32)
print("pos_weight para BCEWithLogitsLoss:", pos_weight.item())

pos_weight para BCEWithLogitsLoss: 68.4246597290039


### Verificación: un batch real, y que el sampler efectivamente balancea

In [6]:
batch = next(iter(train_loader))
print("x_num:", batch["x_num"].shape, batch["x_num"].dtype)
print("x_type:", batch["x_type"].shape, batch["x_type"].dtype)
print("mask:", batch["mask"].shape, batch["mask"].dtype)
print("y:", batch["y"].shape, "proporción positiva en este batch:", batch["y"].mean().item())

# Verificamos que el sampler sube la proporción de positivos en el training loader
# (val/test loaders NO deben tener el sampler -> conservan la proporción natural)
all_y_train_sampled = torch.cat([b["y"] for _, b in zip(range(20), train_loader)])
all_y_val = torch.cat([b["y"] for b in val_loader])
print(f"\nProporción positiva tras {len(all_y_train_sampled)} muestras vía sampler (train): {all_y_train_sampled.mean():.2%}")
print(f"Proporción positiva real en val (sin sampler): {all_y_val.mean():.2%}")

x_num: torch.Size([64, 24, 5]) torch.float32
x_type: torch.Size([64, 24]) torch.int64
mask: torch.Size([64, 24]) torch.float32
y: torch.Size([64]) proporción positiva en este batch: 0.625

Proporción positiva tras 1280 muestras vía sampler (train): 44.77%
Proporción positiva real en val (sin sampler): 1.38%


## Resumen de decisiones — Parte 2

| Decisión | Elección | Motivo |
|---|---|---|
| `oldbalanceDest`, `newbalanceDest`, `errorBalanceDest` | **Excluidas** | Artefacto documentado del simulador (balances de destino no actualizados), separaba fraude/no-fraude de forma artificial y no generalizable |
| `oldbalanceOrg`, `newbalanceOrig`, `errorBalanceOrig` | Se mantienen | Reflejan comportamiento genuino simulado (drenado de cuenta de origen) |
| `isFlaggedFraud` | Excluida | Regla de negocio trivial y casi nunca activada; sin señal aprendible |
| Montos negativos/cero, duplicados | Ninguno encontrado en el sample | Verificado, sin necesidad de limpieza |
| Outliers de `amount` | No se truncan | El log1p ya comprime la cola; montos grandes son señal relevante |
| `type_idx` | Separado del resto de features numéricas | Va a una capa `nn.Embedding` en el modelo, no se normaliza como el resto |
| Desbalance | `WeightedRandomSampler` solo en train | val/test conservan la distribución real para métricas honestas |
